# 08 · XGBoost Deep Dive — servability & the head-to-head

**Phase goal:** close the loop on the model bake-off (nb 05). XGBoost had the *best* cross-validated MAE, yet the pipeline shipped **LightGBM**. This notebook answers the two questions that leaves open:

1. *Why* couldn't XGBoost be shipped through a scikit-learn `Pipeline`?
2. If we serve it another way, **is XGBoost actually better** than the LightGBM we shipped — i.e., did the selection cost us any accuracy?

Full write-up: [`docs/XGBOOST_SERVABILITY.md`](../docs/XGBOOST_SERVABILITY.md).

In [1]:
import sys, warnings
sys.path.insert(0, '../src')
warnings.filterwarnings('ignore')
import numpy as np, pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.metrics import r2_score, mean_absolute_error, mean_squared_error
from xgboost import XGBRegressor
from car_pricing import config, data, features, models
from car_pricing.pipeline import make_pipeline
df = data.clean(data.load_raw()); X, y = features.split_xy(df)
Xtr, Xte, ytr, yte = train_test_split(X, y, test_size=config.TEST_SIZE, random_state=config.RANDOM_STATE)
edges = features.band_edges(ytr)
print('train/test:', len(Xtr), '/', len(Xte))

train/test: 15856 / 3964


## 1 · Reproduce the failure
XGBoost fits inside a `Pipeline` fine — the break is at **`.predict()`**, when scikit-learn 1.6 inspects the estimator's tags.

In [2]:
pipe = make_pipeline(models.model_zoo()['XGBoost'])
pipe.fit(Xtr, ytr)          # fitting is fine
try:
    pipe.predict(Xte[:3])
    print('predict OK (env already patched)')
except Exception as e:
    print('predict FAILS ->', type(e).__name__ + ':', str(e)[:70])

predict FAILS -> AttributeError: 'super' object has no attribute '__sklearn_tags__'


## 2 · Root cause — a broken cooperative-inheritance chain
scikit-learn 1.6 builds estimator tags by walking the MRO and calling `super().__sklearn_tags__()` cooperatively. That requires the **mixin first, `BaseEstimator` last**. XGBoost 2.1.1 declares the bases the other way round, so `RegressorMixin`'s `super()` lands on `object` — which has no `__sklearn_tags__` → `AttributeError`.

In [3]:
mro = [c.__name__ for c in XGBRegressor.__mro__]
print('XGBRegressor MRO:', ' -> '.join(mro))
print('BaseEstimator before RegressorMixin?',
      mro.index('BaseEstimator') < mro.index('RegressorMixin'),
      '(that ordering is the bug)')
import xgboost, sklearn
print('xgboost', xgboost.__version__, '| scikit-learn', sklearn.__version__)

XGBRegressor MRO: XGBRegressor -> XGBModel -> BaseEstimator -> _HTMLDocumentationLinkMixin -> _MetadataRequester -> RegressorMixin -> object
BaseEstimator before RegressorMixin? True (that ordering is the bug)
xgboost 2.1.1 | scikit-learn 1.6.1


### Fix options
| Option | What | Verdict |
| :-- | :-- | :-- |
| **Upgrade** | `pip install 'xgboost>=2.1.2'` (or pin `scikit-learn<1.6`) | ✅ Cleanest — XGBoost then drops straight into the Pipeline |
| **Decoupled serving** | Fit the `ColumnTransformer` + `XGBRegressor` separately; `xgb.predict(pre.transform(X))` | ✅ Works on any versions (used below to evaluate) |
| Monkey-patch tags | Inject a `__sklearn_tags__` at runtime | ⚠️ Fragile / version-specific — not recommended |

## 3 · Serve XGBoost decoupled and evaluate on the held-out test set

In [4]:
pre = features.build_preprocessor()
Xtr_e = pre.fit_transform(Xtr, ytr); Xte_e = pre.transform(Xte)
xgb = XGBRegressor(**models.model_zoo()['XGBoost'].get_params())
xgb.fit(Xtr_e, ytr); pred = xgb.predict(Xte_e)
def score(yt, p):
    tb = features.price_to_band(yt, edges); pb = features.price_to_band(p, edges)
    return {'R2': round(r2_score(yt,p),4), 'MAE': round(mean_absolute_error(yt,p),3),
            'RMSE': round(float(np.sqrt(mean_squared_error(yt,p))),3),
            'band_acc': round(float((tb==pb).mean()),4)}
print('XGBoost (decoupled) TEST:', score(yte, pred))

XGBoost (decoupled) TEST: {'R2': 0.9568, 'MAE': 0.668, 'RMSE': 1.021, 'band_acc': 0.8605}


## 4 · Head-to-head: XGBoost vs the shipped LightGBM

In [5]:
import json
comp = json.loads(config.COMPARISON_PATH.read_text())
m = json.loads(config.METRICS_PATH.read_text())
xgb_test = score(yte, pred)
tbl = pd.DataFrame({
  'XGBoost':  {'CV MAE': round(comp['XGBoost']['cv_mae'],3),
               'Test MAE': xgb_test['MAE'], 'Test R2': xgb_test['R2'],
               'Test RMSE': xgb_test['RMSE'], 'Band acc': xgb_test['band_acc'],
               'Servable in Pipeline': 'no (this env)'},
  'LightGBM (shipped)': {'CV MAE': round(comp['LightGBM']['cv_mae'],3),
               'Test MAE': m['price_mae_lakhs'], 'Test R2': m['price_r2'],
               'Test RMSE': m['price_rmse_lakhs'], 'Band acc': m['band_accuracy'],
               'Servable in Pipeline': 'yes'},
})
tbl

,XGBoost,LightGBM (shipped)
CV MAE,0.708,0.715
Test MAE,0.668,0.664
Test R2,0.9568,0.9568
Test RMSE,1.021,1.022
Band acc,0.8605,0.8582
Servable in Pipeline,no (this env),yes


## Conclusion
- **The models are statistically indistinguishable on the held-out test set** — identical R², MAE within ~₹400, band accuracy within 0.2 pt. Note LightGBM is even *marginally better on test MAE* despite XGBoost's slightly better CV MAE, confirming the gap is noise.
- **Shipping LightGBM therefore cost zero accuracy** while buying clean, shim-free serialisation and serving — exactly the operational-robustness trade-off `train.py` automates.
- **To ship XGBoost instead**, upgrade to `xgboost>=2.1.2`; it then drops straight into the same `Pipeline` and the selector would pick it automatically. Until then, decoupled serving (above) is the escape hatch.

The bake-off thread is now fully closed.